In [ ]:
import pandas as pd
import numpy as np
import os
import itertools
import pandas as pd
import cv2
import glob

np.random.seed(42)

In [ ]:
###############     # infant cats    ###############
chosen_cats = ["bottle", "bowl", "chair", "cup", 
                "door", "spoon", "table", "window"]

In [ ]:
type_comparison = 'all'

cats = ["bottle", "bowl", "chair", "cup", 
                "door", "spoon", "table", "window"]

# load and compute histograms
image_folder = "/path/to/egocentric_images/" # NOTE these will be released to Databrary
histograms = {}

for image_path in glob.glob(os.path.join(image_folder, "*.jpg")):
    filename = os.path.basename(image_path)
    image = cv2.imread(image_path)
    if image is None:
        raise
        # continue
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hist = cv2.calcHist([image_rgb], [0, 1, 2], None, [8, 8, 8],
                        [0, 256, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    histograms[filename] = hist

In [ ]:
df = pd.read_csv("../../PanelA/RGB_hist/NULL_afterSampling.csv")

In [ ]:
rows = []
for cat in cats:

    df_filtered = df [ df['obj'] == cat ].copy()
    frame_ids = sorted(df_filtered['frame_id'].unique())

    # only use frames for which histograms exist
    valid_frames = []
    # only use frames for which histograms exist
    for f in frame_ids:
        if f in histograms: # if doing all to all you can comment out all cats
            valid_frames.append(f)
        else:
            pass
            # print(f)
            # print(len(frame_ids))
            # print(f in jpgs)
    
    if len(valid_frames) < 10:
        raise

    pairs = list(itertools.combinations(valid_frames, 2))

    for img1, img2 in pairs:
        cat1 = cat
        cat2 = cat
        category_pair = f"{cat1}_{cat2}"
        valid = (cat1 == cat2)

        # Compute histogram correlation
        hist_corr = cv2.compareHist(histograms[img1], histograms[img2], cv2.HISTCMP_CORREL)

        rows.append({
            "subj": "noSub_NULL_Unif",
            "img1": img1,
            "img2": img2,
            "category": category_pair,
            "valid": valid,
            "hist_corr": hist_corr
        })
    
    del df_filtered, frame_ids, valid_frames

df_pairs = pd.DataFrame(rows)
print(df_pairs.shape)
df_pairs.to_csv(f"../../PanelA/RGB_hist/NULL_corr_combined.csv", index=False)

In [ ]:
import pandas as pd
###############     # infant cats    ###############
chosen_cats = ["bottle", "bowl", "chair", "cup", 
                "door", "spoon", "table", "window"]

df_data = pd.read_csv(f"../../PanelA/RGB_hist/NULL_corr_combined.csv")  # Columns: frame_id, obj
# print(df_data.head())

for cat in chosen_cats:
    mask = df_data["category"] == f"{cat}_{cat}"
    df = df_data [ mask ].copy()
    print(df.shape)
    df.to_csv(f"../../PanelA/RGB_hist/NULL_forRplots_{cat}.csv", index=False, header=None)
    del df